In [292]:
import sklearn
import torch
import joblib

### Loading and Setting up Dataset

In [271]:
dataset=sklearn.datasets.fetch_covtype()

In [272]:
print(dataset.DESCR)

.. _covtype_dataset:

Forest covertypes
-----------------

The samples in this dataset correspond to 30×30m patches of forest in the US,
collected for the task of predicting each patch's cover type,
i.e. the dominant species of tree.
There are seven covertypes, making this a multiclass classification problem.
Each sample has 54 features, described on the
`dataset's homepage <https://archive.ics.uci.edu/ml/datasets/Covertype>`__.
Some of the features are boolean indicators,
while others are discrete or continuous measurements.

**Data Set Characteristics:**

=================   ============
Classes                        7
Samples total             581012
Dimensionality                54
Features                     int
=================   ============

:func:`sklearn.datasets.fetch_covtype` will load the covertype dataset;
it returns a dictionary-like 'Bunch' object
with the feature matrix in the ``data`` member
and the target values in ``target``. If optional argument 'as_frame' is
se

In [273]:
X=dataset['data']
y=dataset['target']

In [275]:
#scaling
X=(X-X.mean(axis=0))/X.std(axis=0)

y=y-y.min()

In [277]:
#converting to tensor flow objects
X=torch.FloatTensor(X)
y=torch.tensor(y,dtype=torch.uint8)

In [278]:
#Datasets
from torch.utils.data import TensorDataset,random_split,DataLoader
torch.manual_seed(100)

data_set=TensorDataset(X,y)
train_set,valid_set,test_set=random_split(data_set,[0.7,0.12,0.18])
train_loader=DataLoader(train_set,batch_size=32,shuffle=True)
valid_loader=DataLoader(valid_set,batch_size=32)
test_loader=DataLoader(test_set,batch_size=32)

### Creating structure

In [343]:
import torch.nn as nn
class CoverClassifier(nn.Module):
    def __init__(self,n_input,n_hidden_1,n_hidden_2,n_output):
        super().__init__()
        #Edited for vanishing/exploding gradient
        self.mlp=nn.Sequential(
            nn.Linear(n_input,n_hidden_1),
            nn.BatchNorm1d(n_hidden_1),
            nn.ReLU(),
            nn.Linear(n_hidden_1,n_hidden_2),
            nn.BatchNorm1d(n_hidden_2),
            nn.ReLU(),
            nn.Linear(n_hidden_2,n_output)
        )
    def forward(self,X):
        return self.mlp(X)

In [280]:
values,counts=y.unique(return_counts=True)
print(values)
print(counts)

tensor([0, 1, 2, 3, 4, 5, 6], dtype=torch.uint8)
tensor([211840, 283301,  35754,   2747,   9493,  17367,  20510])


In [281]:
weights=counts.sum()/counts
weights_normal=weights/weights.sum()
weights_normal

tensor([0.0077, 0.0058, 0.0457, 0.5949, 0.1721, 0.0941, 0.0797])

### Training and Evaluating

In [282]:
#Defining Parameters
n_input=54
n_output=7
learning_rate=0.002
n_epochs=20
criterion=nn.CrossEntropyLoss(weight=weights_normal)

In [283]:
model=CoverClassifier(n_input=n_input,n_hidden_1=100,n_hidden_2=100,n_output=n_output)
optimizer=torch.optim.SGD(model.parameters(),lr=learning_rate)

In [284]:
def train_model(model,data_loader,optimizer,n_epochs,criterion):
    model.train()
    for epoch in range(n_epochs):
        total_loss=0
        for X,y in data_loader:
            y_pred=model(X)
            loss=criterion(y_pred,y)
            total_loss+=loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            mean_loss=total_loss/len(data_loader)
        print(f'Epoch:{epoch+1}/{n_epochs}, Loss:{mean_loss:4f}')

In [285]:
train_model(model,test_loader,optimizer,n_epochs,criterion)

Epoch:1/20, Loss:1.557742
Epoch:2/20, Loss:1.052646
Epoch:3/20, Loss:0.899698
Epoch:4/20, Loss:0.835249
Epoch:5/20, Loss:0.795188
Epoch:6/20, Loss:0.765765
Epoch:7/20, Loss:0.742595
Epoch:8/20, Loss:0.723391
Epoch:9/20, Loss:0.706944
Epoch:10/20, Loss:0.692958
Epoch:11/20, Loss:0.680672
Epoch:12/20, Loss:0.669996
Epoch:13/20, Loss:0.660531
Epoch:14/20, Loss:0.651830
Epoch:15/20, Loss:0.643951
Epoch:16/20, Loss:0.636853
Epoch:17/20, Loss:0.630207
Epoch:18/20, Loss:0.624056
Epoch:19/20, Loss:0.618208
Epoch:20/20, Loss:0.612793


In [286]:
import torchmetrics as tm
metric=tm.Accuracy(task='multiclass',num_classes=7)

def evaluate(model,data_loader,metric):
    model.eval()
    metric.reset()
    for X,y in data_loader:
        y_pred=model(X)
        metric.update(y_pred,y)
    return metric.compute()

evaluate(model,valid_loader,metric)

tensor(0.6669)

### Finetuning

In [ ]:
#I wll search for learning rate,n_epochs,n_layer_size
import optuna
def objective(trial,train_loader,valid_loader):

    metric=tm.Accuracy(task='multiclass',num_classes=7)

    #Suggesting
    learning_rate=trial.suggest_float('learning_rate',1e-5,1.5e-1,log=True)
    n_epochs=trial.suggest_int('n_epochs',15,65)
    n_hidden_1=trial.suggest_int('n_hidden',50,350)
    n_hidden_2=trial.suggest_int('n_hidden',50,350)#I made an error here,I used "n_hidden" for both layers

    #structure
    model=CoverClassifier(54,n_hidden_1,n_hidden_2,7)
    optimizer=torch.optim.SGD(model.parameters(),learning_rate)
    criterion=nn.CrossEntropyLoss(
        weight=torch.FloatTensor([0.0077, 0.0058, 0.0457, 0.5949, 0.1721, 0.0941, 0.0797])
    )
    #training
    model.train()
    for epoch in range(n_epochs):
        total_loss=0
        for X,y in train_loader:
            y_pred=model(X)
            loss=criterion(y_pred,y)
            total_loss+=loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
        validation_accuracy=evaluate(model,valid_loader,metric)
        trial.report(validation_accuracy,epoch)
        if trial.should_prune():
            return optuna.TrialPruned()
    #evaluation
    validation_accuracy=evaluate(model,valid_loader,metric)
    return validation_accuracy


In [308]:
objective_with_data =lambda trial: objective(trial,train_loader=train_loader,
                                             valid_loader=valid_loader)

In [ ]:
sampler=optuna.samplers.TPESampler(seed=100)
Pruner=optuna.pruners.MedianPruner()
study=optuna.create_study(direction='maximize',sampler=sampler,pruner=Pruner)
study.optimize(objective_with_data,n_trials=15)#I made a mistake while seeting hidden layers
#340m 38.1s

[I 2026-06-08 19:43:26,185] A new study created in memory with name: no-name-0e40a6d5-47f4-49e2-8101-15121e329618
[I 2026-06-08 22:02:29,245] Trial 0 finished with value: 0.7630842924118042 and parameters: {'learning_rate': 0.001859125156649442, 'n_epochs': 29, 'n_hidden': 177}. Best is trial 0 with value: 0.7630842924118042.
[I 2026-06-08 23:15:09,940] Trial 1 finished with value: 0.8144605159759521 and parameters: {'learning_rate': 0.03371803227571075, 'n_epochs': 15, 'n_hidden': 86}. Best is trial 1 with value: 0.8144605159759521.
[I 2026-06-09 00:00:36,034] Trial 2 finished with value: 0.8610318303108215 and parameters: {'learning_rate': 0.006325711628308234, 'n_epochs': 57, 'n_hidden': 91}. Best is trial 2 with value: 0.8610318303108215.
[I 2026-06-09 00:44:06,270] Trial 3 finished with value: 0.8306249380111694 and parameters: {'learning_rate': 0.00252140277319705, 'n_epochs': 60, 'n_hidden': 112}. Best is trial 2 with value: 0.8610318303108215.
[I 2026-06-09 00:57:55,962] Trial 

In [293]:
study.optimize(objective_with_data,n_trials=15)

[W 2026-06-09 01:27:13,629] Trial 15 failed with parameters: {'learning_rate': 0.0018821479486268184, 'n_epochs': 54, 'n_hidden': 125} because of the following error: The value TrialPruned() could not be cast to float.
[W 2026-06-09 01:27:13,630] Trial 15 failed with value TrialPruned().
[W 2026-06-09 01:27:41,182] Trial 16 failed with parameters: {'learning_rate': 0.00015628830952754767, 'n_epochs': 58, 'n_hidden': 343} because of the following error: The value TrialPruned() could not be cast to float.
[W 2026-06-09 01:27:41,183] Trial 16 failed with value TrialPruned().
[W 2026-06-09 01:28:05,436] Trial 17 failed with parameters: {'learning_rate': 0.049571048446422857, 'n_epochs': 33, 'n_hidden': 230} because of the following error: The value TrialPruned() could not be cast to float.
[W 2026-06-09 01:28:05,438] Trial 17 failed with value TrialPruned().
[W 2026-06-09 01:28:27,328] Trial 18 failed with parameters: {'learning_rate': 0.00030315089349363625, 'n_epochs': 32, 'n_hidden': 10

In [294]:
study.optimize(objective_with_data,n_trials=50)

[W 2026-06-09 01:45:44,513] Trial 30 failed with parameters: {'learning_rate': 0.0018062540058561753, 'n_epochs': 30, 'n_hidden': 83} because of the following error: The value TrialPruned() could not be cast to float.
[W 2026-06-09 01:45:44,515] Trial 30 failed with value TrialPruned().
[W 2026-06-09 01:46:18,459] Trial 31 failed with parameters: {'learning_rate': 0.00020212238547051887, 'n_epochs': 38, 'n_hidden': 248} because of the following error: The value TrialPruned() could not be cast to float.
[W 2026-06-09 01:46:18,462] Trial 31 failed with value TrialPruned().
[W 2026-06-09 01:46:50,186] Trial 32 failed with parameters: {'learning_rate': 0.00011529290914159922, 'n_epochs': 47, 'n_hidden': 110} because of the following error: The value TrialPruned() could not be cast to float.
[W 2026-06-09 01:46:50,188] Trial 32 failed with value TrialPruned().
[W 2026-06-09 01:47:27,242] Trial 33 failed with parameters: {'learning_rate': 0.005575723155119913, 'n_epochs': 54, 'n_hidden': 284

In [298]:
study.optimize(objective_with_data,n_trials=100)

[W 2026-06-09 02:55:40,345] Trial 80 failed with parameters: {'learning_rate': 1.4299491649392964e-05, 'n_epochs': 47, 'n_hidden': 334} because of the following error: The value TrialPruned() could not be cast to float.
[W 2026-06-09 02:55:40,346] Trial 80 failed with value TrialPruned().
[W 2026-06-09 02:56:06,030] Trial 81 failed with parameters: {'learning_rate': 1.75616239278417e-05, 'n_epochs': 47, 'n_hidden': 344} because of the following error: The value TrialPruned() could not be cast to float.
[W 2026-06-09 02:56:06,032] Trial 81 failed with value TrialPruned().
[W 2026-06-09 02:56:36,341] Trial 82 failed with parameters: {'learning_rate': 1.1483411014580577e-05, 'n_epochs': 46, 'n_hidden': 348} because of the following error: The value TrialPruned() could not be cast to float.
[W 2026-06-09 02:56:36,342] Trial 82 failed with value TrialPruned().
[W 2026-06-09 02:57:06,732] Trial 83 failed with parameters: {'learning_rate': 1.0001529017254785e-05, 'n_epochs': 46, 'n_hidden': 3

In [ ]:
study.optimize(objective_with_data,n_trials=5,n_jobs=-1)#Disabled pruning and got bad results

[I 2026-06-09 11:27:25,246] Trial 182 finished with value: 0.5769997835159302 and parameters: {'learning_rate': 1.179404547533471e-05, 'n_epochs': 43, 'n_hidden': 347}. Best is trial 35 with value: 0.8978930115699768.
[I 2026-06-09 11:31:45,663] Trial 181 finished with value: 0.6063166260719299 and parameters: {'learning_rate': 1.744753151880358e-05, 'n_epochs': 46, 'n_hidden': 321}. Best is trial 35 with value: 0.8978930115699768.
[I 2026-06-09 11:33:23,081] Trial 184 finished with value: 0.5972662568092346 and parameters: {'learning_rate': 1.660270748780255e-05, 'n_epochs': 47, 'n_hidden': 336}. Best is trial 35 with value: 0.8978930115699768.
[I 2026-06-09 11:34:12,048] Trial 183 finished with value: 0.584486722946167 and parameters: {'learning_rate': 1.2882161927773336e-05, 'n_epochs': 47, 'n_hidden': 347}. Best is trial 35 with value: 0.8978930115699768.
[I 2026-06-09 11:34:38,815] Trial 180 finished with value: 0.5993602871894836 and parameters: {'learning_rate': 1.25449668250110

In [309]:
study.optimize(objective_with_data,n_trials=20)

[W 2026-06-09 11:59:53,853] Trial 185 failed with parameters: {'learning_rate': 0.06902028617974808, 'n_epochs': 25, 'n_hidden': 277} because of the following error: The value TrialPruned() could not be cast to float.
[W 2026-06-09 11:59:53,854] Trial 185 failed with value TrialPruned().
[W 2026-06-09 12:00:16,486] Trial 186 failed with parameters: {'learning_rate': 0.1002930772378216, 'n_epochs': 38, 'n_hidden': 254} because of the following error: The value TrialPruned() could not be cast to float.
[W 2026-06-09 12:00:16,487] Trial 186 failed with value TrialPruned().
[W 2026-06-09 12:00:40,677] Trial 187 failed with parameters: {'learning_rate': 0.11331058030585504, 'n_epochs': 37, 'n_hidden': 269} because of the following error: The value TrialPruned() could not be cast to float.
[W 2026-06-09 12:00:40,678] Trial 187 failed with value TrialPruned().
[W 2026-06-09 12:01:05,955] Trial 188 failed with parameters: {'learning_rate': 0.12400642865815774, 'n_epochs': 38, 'n_hidden': 267} 

In [312]:
print(study.best_params)
print(study.best_value)
print(study.best_trial)

{'learning_rate': 0.03889058275921559, 'n_epochs': 46, 'n_hidden': 345}
0.8978930115699768
FrozenTrial(number=35, state=<TrialState.COMPLETE: 1>, values=[0.8978930115699768], datetime_start=datetime.datetime(2026, 6, 9, 1, 48, 1, 128322), datetime_complete=datetime.datetime(2026, 6, 9, 2, 11, 15, 989716), params={'learning_rate': 0.03889058275921559, 'n_epochs': 46, 'n_hidden': 345}, user_attrs={}, system_attrs={}, intermediate_values={0: 0.6674029231071472, 1: 0.7369802594184875, 2: 0.7505055665969849, 3: 0.8104158043861389, 4: 0.8261929750442505, 5: 0.7888727784156799, 6: 0.8337803483009338, 7: 0.8284878134727478, 8: 0.8455702066421509, 9: 0.8449821472167969, 10: 0.8424291014671326, 11: 0.8474634885787964, 12: 0.8744280934333801, 13: 0.8614190816879272, 14: 0.8757476210594177, 15: 0.8793046474456787, 16: 0.8817285895347595, 17: 0.8794624209403992, 18: 0.877712607383728, 19: 0.8747006058692932, 20: 0.8890434503555298, 21: 0.8715021014213562, 22: 0.8694941401481628, 23: 0.8837366104125

In [317]:
#Saving study
joblib.dump(study,'..\models\model_finetune.pkl')

<>:2: SyntaxWarning: invalid escape sequence '\m'
<>:2: SyntaxWarning: invalid escape sequence '\m'
C:\Users\USER\AppData\Local\Temp\ipykernel_10672\181256816.py:2: SyntaxWarning: invalid escape sequence '\m'
  joblib.dump(study,'..\models\model_finetune.pkl')


['..\\models\\model_finetune.pkl']

### Final Training

In [345]:
torch.manual_seed(100)

best_params=study.best_params
model_final=CoverClassifier(n_input=n_input,n_hidden_1=best_params['n_hidden'],
                            n_hidden_2=best_params['n_hidden'],n_output=n_output)


In [346]:
#He initialization
def use_he_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_uniform_(module.weight)
        nn.init.zeros_(module.bias)
model_final.apply(use_he_init)

CoverClassifier(
  (mlp): Sequential(
    (0): Linear(in_features=54, out_features=345, bias=True)
    (1): BatchNorm1d(345, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Linear(in_features=345, out_features=345, bias=True)
    (4): BatchNorm1d(345, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): Linear(in_features=345, out_features=7, bias=True)
  )
)

In [347]:
optimizer_final=torch.optim.SGD(model_final.parameters(),
                                lr=best_params['learning_rate'])

In [348]:
from torch.utils.data import ConcatDataset

train_valid_set=ConcatDataset([train_set,valid_set])
train_valid_loader=DataLoader(train_valid_set,batch_size=32,shuffle=True)

In [349]:
train_model(model=model_final,data_loader=train_loader,
            optimizer=optimizer_final,n_epochs=best_params['n_epochs'],criterion=criterion)

Epoch:1/46, Loss:0.710745
Epoch:2/46, Loss:0.569852
Epoch:3/46, Loss:0.517423
Epoch:4/46, Loss:0.480353
Epoch:5/46, Loss:0.455851
Epoch:6/46, Loss:0.436405
Epoch:7/46, Loss:0.423051
Epoch:8/46, Loss:0.405805
Epoch:9/46, Loss:0.397105
Epoch:10/46, Loss:0.389264
Epoch:11/46, Loss:0.377032
Epoch:12/46, Loss:0.371128
Epoch:13/46, Loss:0.366305
Epoch:14/46, Loss:0.358242
Epoch:15/46, Loss:0.354519
Epoch:16/46, Loss:0.349429
Epoch:17/46, Loss:0.345833
Epoch:18/46, Loss:0.340272
Epoch:19/46, Loss:0.336286
Epoch:20/46, Loss:0.333059
Epoch:21/46, Loss:0.325746
Epoch:22/46, Loss:0.324844
Epoch:23/46, Loss:0.323129
Epoch:24/46, Loss:0.320847
Epoch:25/46, Loss:0.318401
Epoch:26/46, Loss:0.315105
Epoch:27/46, Loss:0.312885
Epoch:28/46, Loss:0.308977
Epoch:29/46, Loss:0.309090
Epoch:30/46, Loss:0.306663
Epoch:31/46, Loss:0.305339
Epoch:32/46, Loss:0.301652
Epoch:33/46, Loss:0.301533
Epoch:34/46, Loss:0.300404
Epoch:35/46, Loss:0.298090
Epoch:36/46, Loss:0.298999
Epoch:37/46, Loss:0.295891
Epoch:38/4

In [350]:
evaluate(model_final,test_loader,metric)

tensor(0.8680)

In [364]:
#saving model
model_final_data={
    "model_state_dict":model_final.state_dict(),
    "model_hyperparams":{"n_input":n_input,"n_hidden_1":best_params['n_hidden'],
                            'n_hidden_2':best_params['n_hidden'],"n_output":n_output}
}

torch.save(model_final_data,"..\models\cover_model.pt")

<>:8: SyntaxWarning: invalid escape sequence '\m'
<>:8: SyntaxWarning: invalid escape sequence '\m'
C:\Users\USER\AppData\Local\Temp\ipykernel_10672\2675754123.py:8: SyntaxWarning: invalid escape sequence '\m'
  torch.save(model_final_data,"..\models\cover_model.pt")


In [368]:
loaded_data=torch.load("..\models\cover_model.pt",weights_only=True)
new_model=CoverClassifier(**loaded_data['model_hyperparams'])
new_model.load_state_dict(loaded_data['model_state_dict'])

<>:1: SyntaxWarning: invalid escape sequence '\m'
<>:1: SyntaxWarning: invalid escape sequence '\m'
C:\Users\USER\AppData\Local\Temp\ipykernel_10672\2752661028.py:1: SyntaxWarning: invalid escape sequence '\m'
  loaded_data=torch.load("..\models\cover_model.pt",weights_only=True)


<All keys matched successfully>